# Topic 2: Random Forests
### DS + MLE Interview Study Plan — Theory Notebook

---

## Section 1: What is it?

A random forest is a collection of decision trees that work together. Each tree is trained on a slightly different version of the data using a random subset of features, and the final prediction comes from combining all the trees' answers.

- **Classification:** Each tree votes, majority wins.
- **Regression:** Each tree predicts a number, you take the average.

**The core insight:** A single decision tree is fragile and overfits. But if you build hundreds of *different* trees and average them, the individual errors cancel out and the real patterns survive.

**Two sources of randomness make the trees different:**
1. **Bagging (bootstrap aggregating):** Each tree trains on a random sample of the data, drawn with replacement. Some data points appear multiple times, some don't appear at all.
2. **Feature randomness:** At every split, each tree only considers a random subset of features. This forces trees to find different patterns instead of all relying on the same dominant feature.


## Section 2: Why does it matter?

**When you'd use a random forest:**
- Strong baseline for almost any tabular/structured data problem
- When you want something reliable without heavy hyperparameter tuning
- When you want feature importances to understand what's driving predictions
- When you need a benchmark to compare more complex models against

**When you wouldn't:**
- When you need the absolute best predictive performance — gradient boosting (XGBoost, LightGBM) usually wins
- When interpretability of individual predictions matters — you can't "read" 500 trees like you can read one
- For image, text, or audio data — neural networks are better suited
- When training speed matters at scale — 500 trees take time

**Random forest vs. single decision tree:**
- Much lower variance (less overfitting)
- Better generalization to new data
- More stable — small data changes don't restructure the entire model
- But: you lose the ability to read one simple set of rules

**Random forest vs. gradient boosting (preview):**
- Random forests build trees independently (in parallel) — each tree ignores the others
- Gradient boosting builds trees sequentially — each tree corrects the previous one's mistakes
- Gradient boosting usually performs better but requires more careful tuning
- Random forests are harder to mess up — a good "set and forget" option


## Section 3: How it works — Code Walkthrough

We'll use the same F1-inspired dataset from the decision tree notebook and compare a single tree to a random forest.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

np.random.seed(42)


### 3a. Create the F1 dataset (same as decision tree notebook)


In [ ]:
n = 500

grid_position = np.random.randint(1, 21, n)
team_strength = np.random.uniform(0.3, 1.0, n)
is_wet = np.random.binomial(1, 0.2, n)
tire_compound = np.random.choice([0, 1, 2], n)

podium_prob = 1 / (1 + np.exp(0.5 * grid_position - 5 * team_strength - 0.3 * is_wet + 2))
podium = (np.random.random(n) < podium_prob).astype(int)

df = pd.DataFrame({
    'grid_position': grid_position,
    'team_strength': team_strength,
    'is_wet': is_wet,
    'tire_compound': tire_compound,
    'podium': podium
})

X = df[['grid_position', 'team_strength', 'is_wet', 'tire_compound']]
y = df['podium']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")
print(f"Podium rate: {y.mean():.1%}")


### 3b. Single tree vs. random forest — head to head


In [ ]:
# Single decision tree
single_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
single_tree.fit(X_train, y_train)

# Random forest — 200 trees, each with max_depth=5
forest = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
forest.fit(X_train, y_train)

# Compare
models = {
    'Single Decision Tree': single_tree,
    'Random Forest (200 trees)': forest
}

print("Model Comparison")
print("=" * 55)
for name, model in models.items():
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    gap = train_acc - test_acc
    print(f"\n{name}:")
    print(f"  Train accuracy: {train_acc:.3f}")
    print(f"  Test accuracy:  {test_acc:.3f}")
    print(f"  Gap:            {gap:.3f}  {'⚠️ possible overfitting' if gap > 0.05 else '✓ looks good'}")

print("\n" + "=" * 55)
print("KEY INSIGHT: The random forest should have a smaller gap")
print("between train and test accuracy. That's reduced overfitting.")


### 3c. Feature importances — single tree vs. forest

One of the most useful things about random forests: they tell you which features matter. And because the importance is averaged across hundreds of trees, it's more stable than a single tree's importances.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, models.items()):
    importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()
    importances.plot(kind='barh', ax=ax)
    ax.set_title(f'Feature Importance: {name}')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

print("COMPARE: The random forest importances are usually more")
print("'spread out' and stable. The single tree might put all")
print("its weight on one feature because of its greedy nature.")


## Section 4: Experiments

### Experiment 1: How many trees is enough?

More trees = better performance... up to a point. Let's find where the gains plateau.


In [ ]:
tree_counts = [1, 5, 10, 25, 50, 100, 200, 500]
train_scores = []
test_scores = []

for n_trees in tree_counts:
    rf = RandomForestClassifier(n_estimators=n_trees, max_depth=5, random_state=42)
    rf.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, rf.predict(X_train)))
    test_scores.append(accuracy_score(y_test, rf.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(tree_counts, train_scores, 'o-', label='Train accuracy', color='steelblue')
plt.plot(tree_counts, test_scores, 's-', label='Test accuracy', color='coral')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('How many trees do you need?')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.show()

print("KEY INSIGHT: Test accuracy usually improves quickly at first,")
print("then plateaus. More trees rarely hurts (just costs compute),")
print("but after ~100-200 you often get diminishing returns.")


### Experiment 2: Proving that random forests are more stable than single trees

We'll train multiple models on slightly different samples and see how much the predictions vary.


In [ ]:
n_trials = 10
single_tree_preds = []
forest_preds = []

for i in range(n_trials):
    # Different random sample each time
    sample = df.sample(frac=0.8, random_state=i)
    X_s = sample[X.columns]
    y_s = sample['podium']
    
    # Single tree
    st = DecisionTreeClassifier(max_depth=5, random_state=42)
    st.fit(X_s, y_s)
    single_tree_preds.append(st.predict(X_test))
    
    # Random forest
    rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf.fit(X_s, y_s)
    forest_preds.append(rf.predict(X_test))

# How much do predictions vary across trials?
single_tree_variance = np.array(single_tree_preds).std(axis=0).mean()
forest_variance = np.array(forest_preds).std(axis=0).mean()

print("Prediction Stability (lower = more stable)")
print("=" * 45)
print(f"Single Decision Tree:  {single_tree_variance:.4f}")
print(f"Random Forest:         {forest_variance:.4f}")
print()
if forest_variance < single_tree_variance:
    ratio = single_tree_variance / forest_variance
    print(f"The random forest is {ratio:.1f}x more stable!")
    print("This is the 'variance reduction' in action.")
else:
    print("Results are similar — try with a noisier dataset.")


### Experiment 3: What does max_features control?

`max_features` controls how many features each tree can consider at each split. Lower values = more randomness between trees = more diversity in the ensemble.


In [ ]:
max_features_options = [1, 2, 3, 4]  # Out of 4 total features
results = []

for mf in max_features_options:
    rf = RandomForestClassifier(n_estimators=200, max_depth=5, 
                                 max_features=mf, random_state=42)
    rf.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, rf.predict(X_train))
    test_acc = accuracy_score(y_test, rf.predict(X_test))
    results.append({
        'max_features': mf,
        'train_accuracy': f"{train_acc:.3f}",
        'test_accuracy': f"{test_acc:.3f}",
        'gap': f"{train_acc - test_acc:.3f}"
    })

print("Effect of max_features on performance")
print("=" * 55)
print(pd.DataFrame(results).to_string(index=False))
print()
print("KEY INSIGHT:")
print("- max_features=4 means every tree sees all features (less random)")
print("- max_features=1 means each split picks from just 1 feature (very random)")
print("- The sweet spot is usually sqrt(n_features) for classification")
print(f"- For our 4 features, that's sqrt(4) = 2")


### Experiment 4: Out-of-bag (OOB) score — free validation

Because each tree only sees ~63% of the data (due to bootstrapping), the remaining ~37% can be used as a built-in validation set. This is called the "out-of-bag" score and it's a nice freebie.


In [ ]:
rf_oob = RandomForestClassifier(n_estimators=200, max_depth=5, 
                                 oob_score=True, random_state=42)
rf_oob.fit(X_train, y_train)

test_acc = accuracy_score(y_test, rf_oob.predict(X_test))

print(f"OOB score (free validation):  {rf_oob.oob_score_:.3f}")
print(f"Actual test accuracy:         {test_acc:.3f}")
print()
print("KEY INSIGHT: The OOB score closely approximates test performance")
print("without needing a separate validation set. This is useful when")
print("your dataset is small and you don't want to 'waste' data on a")
print("holdout set.")


## Section 5: My Interview Answer

> **"Explain how a random forest works and why it's better than a single decision tree."**

*"A random forest is an ensemble of decision trees, where each tree is trained on a bootstrapped sample of the data and only considers a random subset of features at each split. The final prediction is the majority vote (classification) or average (regression) across all trees.*

*The key advantage over a single tree is reduced variance. A single tree is fragile — small data changes can completely restructure it, and it tends to overfit. By building hundreds of diverse trees and averaging them, the individual errors cancel out while the real signal survives.*

*The main tradeoff is interpretability. You can't read a forest like you can read one tree, so you rely on feature importances or SHAP values to understand what's driving predictions.*

*In practice, random forests are a great first model for any tabular problem — they're hard to mess up, require minimal tuning, and give you a strong baseline. I'd typically start here, then try gradient boosting to see if I can squeeze out better performance."*

---

## Key Takeaways

1. A random forest builds many decision trees, each on a random subset of data and features, then combines their predictions.
2. Two sources of randomness: bagging (random data samples) and feature subsampling (random features per split).
3. The main benefit is variance reduction — much less overfitting than a single tree.
4. Key hyperparameters: n_estimators (number of trees), max_features (randomness per split), max_depth.
5. Tradeoff: better performance, but less interpretable than a single tree.
6. Random forests build trees independently (parallel). Gradient boosting builds them sequentially (each corrects the last) — that's the next topic.

---

*Previous notebook: Topic 1 — Decision Trees*
*Next notebook: Topic 3 — Gradient Boosting & XGBoost*
